# Natural Gas

In [ ]:
import pandas as pd
import statsmodels.api as sm
from collections import OrderedDict

file_path = "DistilRoBERTa FinBERT LLM emotion .xlsx"
df = pd.read_excel(file_path)

y_var = "natural_gas_return_t"
lag_var = "natural_gas_return_t-1"

finbert_vars = [
    "FinBERT_negative",
    "FinBERT_positive"
]

llm_vars = [
    "LLM_fear",
    "LLM_anger",
    "LLM_disgust",
    "LLM_joy",
    "LLM_sadness",
    "LLM_surprise"
]

distil_vars = [
    "DistilRoBERTa_fear",
    "DistilRoBERTa_anger",
    "DistilRoBERTa_disgust",
    "DistilRoBERTa_joy",
    "DistilRoBERTa_sadness",
    "DistilRoBERTa_surprise"
]

all_regressors = finbert_vars + llm_vars + distil_vars + [lag_var]

def stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""


def run_hac_model(dataframe, y, x_vars, with_lag=True, lag=None, maxlags=1):
    regressors = x_vars.copy()
    if with_lag:
        regressors = regressors + [lag]

    temp = dataframe[[y] + regressors].copy().dropna()

    Y = temp[y]
    X = sm.add_constant(temp[regressors])

    model = sm.OLS(Y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags},
        use_t=True
    )
    return model


def build_regression_table(specs, dataframe, y, lag, maxlags=1):
    row_labels = []
    row_keys = []

    for var in ["const"] + all_regressors:
        pretty_name = "Constant" if var == "const" else var
        row_labels.append(pretty_name)
        row_keys.append((var, "coef"))
        row_labels.append("")
        row_keys.append((var, "se"))

    row_labels.extend(["N", "Adj. R-squared"])
    row_keys.extend([("N", "stat"), ("Adj. R-squared", "stat")])

    output = pd.DataFrame({"Variable": row_labels})

    for model_name, (x_vars, with_lag) in specs.items():
        model = run_hac_model(
            dataframe=dataframe,
            y=y,
            x_vars=x_vars,
            with_lag=with_lag,
            lag=lag,
            maxlags=maxlags
        )

        col_values = []
        for key, kind in row_keys:
            if kind == "coef":
                if key in model.params.index:
                    col_values.append(f"{model.params[key]:.4f}{stars(model.pvalues[key])}")
                else:
                    col_values.append("")
            elif kind == "se":
                if key in model.bse.index:
                    col_values.append(f'="({model.bse[key]:.4f})"')
                else:
                    col_values.append("")
            elif kind == "stat":
                if key == "N":
                    col_values.append(int(model.nobs))
                elif key == "Adj. R-squared":
                    col_values.append(f"{model.rsquared_adj:.4f}")

        output[model_name] = col_values

    return output



In [37]:
main_specs = OrderedDict([
    ("FinBERT Only (with t-1)", (finbert_vars, True)),
    ("FinBERT Only (without t-1)", (finbert_vars, False)),

    ("6LLM Only (with t-1)", (llm_vars, True)),
    ("6LLM Only (without t-1)", (llm_vars, False)),

    ("DistilRoBERTa Only (with t-1)", (distil_vars, True)),
    ("DistilRoBERTa Only (without t-1)", (distil_vars, False)),

    ("FinBERT + 6LLM (with t-1)", (finbert_vars + llm_vars, True)),
    ("FinBERT + 6LLM (without t-1)", (finbert_vars + llm_vars, False)),

    ("LLM + DistilRoBERTa (with t-1)", (llm_vars + distil_vars, True)),
    ("LLM + DistilRoBERTa (without t-1)", (llm_vars + distil_vars, False)),

    ("FinBERT + 6LLM + 6DistilRoBERTa (with t-1)", (finbert_vars + llm_vars + distil_vars, True)),
    ("FinBERT + 6LLM + 6DistilRoBERTa (without t-1)", (finbert_vars + llm_vars + distil_vars, False)),

    ("FinBERT + DistilRoBERTa (with t-1)", (finbert_vars + distil_vars, True)),
    ("FinBERT + DistilRoBERTa (without t-1)", (finbert_vars + distil_vars, False)),
])

llm_each_specs = OrderedDict()
for emo in llm_vars:
    short = emo.replace("LLM_", "").capitalize()
    llm_each_specs[f"LLM {short} (with t-1)"] = ([emo], True)
    llm_each_specs[f"LLM {short} (without t-1)"] = ([emo], False)

llm_each_finbert_specs = OrderedDict()
for emo in llm_vars:
    short = emo.replace("LLM_", "").capitalize()
    llm_each_finbert_specs[f"LLM {short} + FinBERT (with t-1)"] = ([emo] + finbert_vars, True)
    llm_each_finbert_specs[f"LLM {short} + FinBERT (without t-1)"] = ([emo] + finbert_vars, False)

In [38]:
main_table = build_regression_table(
    specs=main_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

llm_each_table = build_regression_table(
    specs=llm_each_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

llm_each_finbert_table = build_regression_table(
    specs=llm_each_finbert_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

all_in_one = pd.concat(
    [
        main_table,
        llm_each_table.drop(columns=["Variable"]),
        llm_each_finbert_table.drop(columns=["Variable"])
    ],
    axis=1
)

all_in_one.to_clipboard(excel=True, index=False)
print(all_in_one)

                  Variable FinBERT Only (with t-1) FinBERT Only (without t-1)  \
0                 Constant                 -0.0189                    -0.0223   
1                                      ="(0.0321)"                ="(0.0322)"   
2         FinBERT_negative                  0.3271                     0.3155   
3                                      ="(0.3471)"                ="(0.3383)"   
4         FinBERT_positive                  0.0213                     0.0454   
5                                      ="(0.1253)"                ="(0.1257)"   
6                 LLM_fear                                                      
7                                                                               
8                LLM_anger                                                      
9                                                                               
10             LLM_disgust                                                      
11                          

# Oil

In [ ]:
import pandas as pd
import statsmodels.api as sm
from collections import OrderedDict

file_path = "DistilRoBERTa FinBERT LLM emotion .xlsx"
df = pd.read_excel(file_path)

y_var = "oil_return_t"
lag_var = "oil_return_t-1"

finbert_vars = [
    "FinBERT_negative",
    "FinBERT_positive"
]

llm_vars = [
    "LLM_fear",
    "LLM_anger",
    "LLM_disgust",
    "LLM_joy",
    "LLM_sadness",
    "LLM_surprise"
]

distil_vars = [
    "DistilRoBERTa_fear",
    "DistilRoBERTa_anger",
    "DistilRoBERTa_disgust",
    "DistilRoBERTa_joy",
    "DistilRoBERTa_sadness",
    "DistilRoBERTa_surprise"
]

all_regressors = finbert_vars + llm_vars + distil_vars + [lag_var]

def stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""


def run_hac_model(dataframe, y, x_vars, with_lag=True, lag=None, maxlags=1):
    regressors = x_vars.copy()
    if with_lag:
        regressors = regressors + [lag]

    temp = dataframe[[y] + regressors].copy().dropna()

    Y = temp[y]
    X = sm.add_constant(temp[regressors])

    model = sm.OLS(Y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags},
        use_t=True
    )
    return model


def build_regression_table(specs, dataframe, y, lag, maxlags=1):
    row_labels = []
    row_keys = []

    for var in ["const"] + all_regressors:
        pretty_name = "Constant" if var == "const" else var
        row_labels.append(pretty_name)
        row_keys.append((var, "coef"))
        row_labels.append("")
        row_keys.append((var, "se"))

    row_labels.extend(["N", "Adj. R-squared"])
    row_keys.extend([("N", "stat"), ("Adj. R-squared", "stat")])

    output = pd.DataFrame({"Variable": row_labels})

    for model_name, (x_vars, with_lag) in specs.items():
        model = run_hac_model(
            dataframe=dataframe,
            y=y,
            x_vars=x_vars,
            with_lag=with_lag,
            lag=lag,
            maxlags=maxlags
        )

        col_values = []
        for key, kind in row_keys:
            if kind == "coef":
                if key in model.params.index:
                    col_values.append(f"{model.params[key]:.4f}{stars(model.pvalues[key])}")
                else:
                    col_values.append("")
            elif kind == "se":
                if key in model.bse.index:
                    col_values.append(f'="({model.bse[key]:.4f})"')
                else:
                    col_values.append("")
            elif kind == "stat":
                if key == "N":
                    col_values.append(int(model.nobs))
                elif key == "Adj. R-squared":
                    col_values.append(f"{model.rsquared_adj:.4f}")

        output[model_name] = col_values

    return output





In [41]:
main_specs = OrderedDict([
    ("FinBERT Only (with t-1)", (["FinBERT"], True)),
    ("FinBERT Only (without t-1)", (["FinBERT"], False)),

    ("6LLM Only (with t-1)", (llm_vars, True)),
    ("6LLM Only (without t-1)", (llm_vars, False)),

    ("DistilRoBERTa Only (with t-1)", (distil_vars, True)),
    ("DistilRoBERTa Only (without t-1)", (distil_vars, False)),

    ("FinBERT + 6LLM (with t-1)", (["FinBERT"] + llm_vars, True)),
    ("FinBERT + 6LLM (without t-1)", (["FinBERT"] + llm_vars, False)),

    ("LLM + DistilRoBERTa (with t-1)", (llm_vars + distil_vars, True)),
    ("LLM + DistilRoBERTa (without t-1)", (llm_vars + distil_vars, False)),

    ("FinBERT + 6LLM + 6DistilRoBERTa (with t-1)", (["FinBERT"] + llm_vars + distil_vars, True)),
    ("FinBERT + 6LLM + 6DistilRoBERTa (without t-1)", (["FinBERT"] + llm_vars + distil_vars, False)),

    ("FinBERT + DistilRoBERTa (with t-1)", (["FinBERT"] + distil_vars, True)),
    ("FinBERT + DistilRoBERTa (without t-1)", (["FinBERT"] + distil_vars, False)),
])

llm_each_specs = OrderedDict()
for emo in llm_vars:
    short = emo.replace("LLM_", "").capitalize()
    llm_each_specs[f"LLM {short} (with t-1)"] = ([emo], True)
    llm_each_specs[f"LLM {short} (without t-1)"] = ([emo], False)

llm_each_finbert_specs = OrderedDict()
for emo in llm_vars:
    short = emo.replace("LLM_", "").capitalize()
    llm_each_finbert_specs[f"LLM {short} + FinBERT (with t-1)"] = ([emo, "FinBERT"], True)
    llm_each_finbert_specs[f"LLM {short} + FinBERT (without t-1)"] = ([emo, "FinBERT"], False)


In [42]:
main_table = build_regression_table(
    specs=main_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

llm_each_table = build_regression_table(
    specs=llm_each_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

llm_each_finbert_table = build_regression_table(
    specs=llm_each_finbert_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

all_in_one = pd.concat(
    [
        main_table,
        llm_each_table.drop(columns=["Variable"]),
        llm_each_finbert_table.drop(columns=["Variable"])
    ],
    axis=1
)


all_in_one.to_clipboard(excel=True, index=False)
print(all_in_one)


                  Variable FinBERT Only (with t-1) FinBERT Only (without t-1)  \
0                 Constant                 -0.0100                    -0.0114   
1                                      ="(0.0095)"                ="(0.0094)"   
2         FinBERT_negative                                                      
3                                                                               
4         FinBERT_positive                                                      
5                                                                               
6                 LLM_fear                                                      
7                                                                               
8                LLM_anger                                                      
9                                                                               
10             LLM_disgust                                                      
11                          

# Clean Energy

In [ ]:
import pandas as pd
import statsmodels.api as sm
from collections import OrderedDict

file_path = "DistilRoBERTa FinBERT LLM emotion .xlsx"
df = pd.read_excel(file_path)

y_var = "clean_return_t"
lag_var = "clean_return_t-1"

finbert_vars = [
    "FinBERT_negative",
    "FinBERT_positive"
]

llm_vars = [
    "LLM_fear",
    "LLM_anger",
    "LLM_disgust",
    "LLM_joy",
    "LLM_sadness",
    "LLM_surprise"
]

distil_vars = [
    "DistilRoBERTa_fear",
    "DistilRoBERTa_anger",
    "DistilRoBERTa_disgust",
    "DistilRoBERTa_joy",
    "DistilRoBERTa_sadness",
    "DistilRoBERTa_surprise"
]

all_regressors = finbert_vars + llm_vars + distil_vars + [lag_var]

def stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""


def run_hac_model(dataframe, y, x_vars, with_lag=True, lag=None, maxlags=1):
    regressors = x_vars.copy()
    if with_lag:
        regressors = regressors + [lag]

    temp = dataframe[[y] + regressors].copy().dropna()

    Y = temp[y]
    X = sm.add_constant(temp[regressors])

    model = sm.OLS(Y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags},
        use_t=True
    )
    return model


def build_regression_table(specs, dataframe, y, lag, maxlags=1):
    row_labels = []
    row_keys = []

    for var in ["const"] + all_regressors:
        pretty_name = "Constant" if var == "const" else var
        row_labels.append(pretty_name)
        row_keys.append((var, "coef"))
        row_labels.append("")
        row_keys.append((var, "se"))

    row_labels.extend(["N", "Adj. R-squared"])
    row_keys.extend([("N", "stat"), ("Adj. R-squared", "stat")])

    output = pd.DataFrame({"Variable": row_labels})

    for model_name, (x_vars, with_lag) in specs.items():
        model = run_hac_model(
            dataframe=dataframe,
            y=y,
            x_vars=x_vars,
            with_lag=with_lag,
            lag=lag,
            maxlags=maxlags
        )

        col_values = []
        for key, kind in row_keys:
            if kind == "coef":
                if key in model.params.index:
                    col_values.append(f"{model.params[key]:.4f}{stars(model.pvalues[key])}")
                else:
                    col_values.append("")
            elif kind == "se":
                if key in model.bse.index:
                    col_values.append(f'="({model.bse[key]:.4f})"')
                else:
                    col_values.append("")
            elif kind == "stat":
                if key == "N":
                    col_values.append(int(model.nobs))
                elif key == "Adj. R-squared":
                    col_values.append(f"{model.rsquared_adj:.4f}")

        output[model_name] = col_values

    return output





In [44]:
main_specs = OrderedDict([
    ("FinBERT Only (with t-1)", (finbert_vars, True)),
    ("FinBERT Only (without t-1)", (finbert_vars, False)),

    ("6LLM Only (with t-1)", (llm_vars, True)),
    ("6LLM Only (without t-1)", (llm_vars, False)),

    ("DistilRoBERTa Only (with t-1)", (distil_vars, True)),
    ("DistilRoBERTa Only (without t-1)", (distil_vars, False)),

    ("FinBERT + 6LLM (with t-1)", (finbert_vars + llm_vars, True)),
    ("FinBERT + 6LLM (without t-1)", (finbert_vars + llm_vars, False)),

    ("LLM + DistilRoBERTa (with t-1)", (llm_vars + distil_vars, True)),
    ("LLM + DistilRoBERTa (without t-1)", (llm_vars + distil_vars, False)),

    ("FinBERT + 6LLM + 6DistilRoBERTa (with t-1)", (finbert_vars + llm_vars + distil_vars, True)),
    ("FinBERT + 6LLM + 6DistilRoBERTa (without t-1)", (finbert_vars + llm_vars + distil_vars, False)),

    ("FinBERT + DistilRoBERTa (with t-1)", (finbert_vars + distil_vars, True)),
    ("FinBERT + DistilRoBERTa (without t-1)", (finbert_vars + distil_vars, False)),
])

llm_each_specs = OrderedDict()
for emo in llm_vars:
    short = emo.replace("LLM_", "").capitalize()
    llm_each_specs[f"LLM {short} (with t-1)"] = ([emo], True)
    llm_each_specs[f"LLM {short} (without t-1)"] = ([emo], False)

llm_each_finbert_specs = OrderedDict()
for emo in llm_vars:
    short = emo.replace("LLM_", "").capitalize()
    llm_each_finbert_specs[f"LLM {short} + FinBERT (with t-1)"] = ([emo] + finbert_vars, True)
    llm_each_finbert_specs[f"LLM {short} + FinBERT (without t-1)"] = ([emo] + finbert_vars, False)

In [45]:
main_table = build_regression_table(
    specs=main_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

llm_each_table = build_regression_table(
    specs=llm_each_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

llm_each_finbert_table = build_regression_table(
    specs=llm_each_finbert_specs,
    dataframe=df,
    y=y_var,
    lag=lag_var,
    maxlags=1
)

all_in_one = pd.concat(
    [
        main_table,
        llm_each_table.drop(columns=["Variable"]),
        llm_each_finbert_table.drop(columns=["Variable"])
    ],
    axis=1
)


all_in_one.to_clipboard(excel=True, index=False)
print(all_in_one)


                  Variable FinBERT Only (with t-1) FinBERT Only (without t-1)  \
0                 Constant                 -0.0136                    -0.0150   
1                                      ="(0.0152)"                ="(0.0149)"   
2         FinBERT_negative                  0.1447                     0.1534   
3                                      ="(0.1505)"                ="(0.1524)"   
4         FinBERT_positive                  0.0648                     0.0733   
5                                      ="(0.0662)"                ="(0.0627)"   
6                 LLM_fear                                                      
7                                                                               
8                LLM_anger                                                      
9                                                                               
10             LLM_disgust                                                      
11                          